### Data Quality Checks

Purpose:
- Validate dbt-generated models
- Detect data quality issues
- Ensure analytical tables are reliable

dbt Models Tested:
- models/staging/stg_customer_support_tickets.sql
- models/marts/dim_customers.sql
- models/marts/dim_products.sql
- models/marts/int_ticket_performance.sql
- models/marts/fact_ticket_metrics.sql

Import libraries

In [1]:
import duckdb
import pandas as pd
import numpy as np

from datetime import datetime

Connect to DuckDB

In [2]:
conn = duckdb.connect(
    "../customer_support.duckdb",
    read_only=True
)

print("DuckDB connection successful")

DuckDB connection successful


Create Query Helper Function

In [3]:
def run_query(query):
    return conn.execute(query).df()

Show available tables

In [4]:
run_query("""
SHOW TABLES;
""")

,name
0,dim_customers
1,dim_products
2,fact_ticket_metrics
3,int_ticket_performance
4,stg_customer_support_tickets


Row counts check

In [5]:
run_query("""
SELECT
    'stg_customer_support_tickets' AS table_name,
    COUNT(*) AS records
FROM main.stg_customer_support_tickets

UNION ALL

SELECT
    'dim_customers',
    COUNT(*)
FROM main.dim_customers

UNION ALL

SELECT
    'dim_products',
    COUNT(*)
FROM main.dim_products

UNION ALL

SELECT
    'fact_ticket_metrics',
    COUNT(*)
FROM main.fact_ticket_metrics;
""")

,table_name,records
0,stg_customer_support_tickets,8469
1,dim_customers,8469
2,dim_products,42
3,fact_ticket_metrics,8791


Null values check

In [6]:
run_query("""
SELECT
    COUNT(*) AS total_records,

    SUM(
        CASE WHEN customer_id IS NULL 
        THEN 1 ELSE 0 END
    ) AS missing_customer_id,

    SUM(
        CASE WHEN customer_email IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_email

FROM main.dim_customers;
""")

,total_records,missing_customer_id,missing_email
0,8469,0.0,0.0


Duplicate checks for Customers

In [8]:
run_query("""
SELECT
    customer_id,
    COUNT(*) AS duplicate_count

FROM main.dim_customers

GROUP BY customer_id

HAVING COUNT(*) > 1;
""")

,customer_id,duplicate_count


Duplicate checks for Tickets

In [9]:
run_query("""
SELECT
    ticket_id,
    COUNT(*) AS duplicate_count

FROM main.stg_customer_support_tickets

GROUP BY ticket_id

HAVING COUNT(*) > 1;
""")

,ticket_id,duplicate_count


Referential Integrity

Check that fact tables correctly link to dimensions.

In [10]:
run_query("""
SELECT COUNT(*) AS unmatched_customers

FROM main.fact_ticket_metrics f

LEFT JOIN main.dim_customers c
ON f.customer_id = c.customer_id

WHERE c.customer_id IS NULL;
""")

,unmatched_customers
0,0


Check customer age validity:

In [11]:
run_query("""
SELECT
    MIN(customer_age) AS minimum_age,
    MAX(customer_age) AS maximum_age

FROM main.dim_customers;
""")

,minimum_age,maximum_age
0,18,70


Customer satisfaction must be between 1 and 5

In [12]:
run_query("""
SELECT DISTINCT
    customer_satisfaction_rating

FROM main.fact_ticket_metrics

ORDER BY 1;
""")

,customer_satisfaction_rating
0,1.0
1,2.0
2,3.0
3,4.0
4,5.0
5,NaN


Product Validation

In [13]:
run_query("""
SELECT
    COUNT(*) AS products,
    COUNT(DISTINCT product_name) AS unique_products

FROM main.dim_products;
""")

,products,unique_products
0,42,42


Close connection

In [14]:
conn.close()

print("Connection closed")

Connection closed
